# Расчёт осевого компрессора — по формулам РПЗ

Данный файл воспроизводит **все формулы** из `rpz.md` (нумерация сохранена).
Для каждой формулы выводятся **все входные переменные** и **подставленные значения**.

## 0. Газодинамические функции

In [ ]:
import numpy as np
from scipy.optimize import brentq
import warnings
warnings.filterwarnings('ignore')

def tau_lam(lam, k=1.4):
    return 1.0 - (k - 1.0) / (k + 1.0) * lam**2

def eps_lam(lam, k=1.4):
    t = tau_lam(lam, k)
    if isinstance(t, np.ndarray):
        return np.where(t > 0, np.abs(t)**(1.0/(k-1.0)), 0.0)
    return t**(1.0/(k-1.0)) if t > 0 else 0.0

def q_lam(lam, k=1.4):
    t = tau_lam(lam, k)
    if isinstance(t, np.ndarray):
        return np.where(t > 0, lam * ((k+1)/2)**(1/(k-1)) * np.abs(t)**(1/(k-1)), 0.0)
    if t <= 0: return 0.0
    return lam * ((k+1)/2)**(1/(k-1)) * t**(1/(k-1))

def beta_coef(k=1.4):
    return np.sqrt(k) * (2.0/(k+1.0))**((k+1.0)/(2.0*(k-1.0)))

def a_cr_func(T_star, k=1.4, Rg=287.4):
    return np.sqrt(2.0*k/(k+1.0) * Rg * T_star)

_CP_COEFFS = np.array([1.087192, -0.684074, 1.859839, -1.8104, 0.89029, -0.221403, 0.022161])

def cp_air(T, Rg=287.4):
    return np.polyval(_CP_COEFFS[::-1], T / 1000.0) * 1000.0

def k_air(T, Rg=287.4):
    cp = cp_air(T, Rg)
    return cp / (cp - Rg)

def eta_stage_poly(ca_bar):
    return -2.454 * ca_bar**2 + 2.401 * ca_bar + 0.296

def HT_bar_poly(ca_bar):
    return -1.015 * ca_bar**2 + 0.408 * ca_bar + 0.344

print("Функции загружены.")

## 1. Исходные данные

In [ ]:
G = 58.7;  T0_star = 288.0;  P0_star = 1.013e5
pi_k_star = 18.2;  n_rpm = 8600;  Rg = 287.4
k0 = k_air(T0_star, Rg);  cp0 = cp_air(T0_star, Rg)

print("=== Техническое задание ===")
print(f"  G       = {G} кг/с")
print(f"  T*a     = {T0_star} К")
print(f"  P*a     = {P0_star:.0f} Па")
print(f"  pi*k    = {pi_k_star}")
print(f"  n       = {n_rpm} об/мин")
print(f"  Rg      = {Rg} Дж/(кг*К)")
print(f"  k(T*a)  = {k0:.4f}")
print(f"  cp(T*a) = {cp0:.1f} Дж/(кг*К)")

## 1.1 Выбор основных параметров

In [ ]:
ca_bar_1 = 0.4884;  d_bar_1 = 0.5500;  R_sr_base = 0.5
zeta_vkh = 0.05;  zeta_vykh = 0.10;  eta_pol = 0.90

print("=== Выбранные параметры ===")
print(f"  ca_bar_1  = {ca_bar_1}")
print(f"  d_bar_1   = {d_bar_1}")
print(f"  R_sr      = {R_sr_base}")
print(f"  zeta_vkh  = {zeta_vkh}")
print(f"  zeta_vykh = {zeta_vykh}")
print(f"  eta_pol   = {eta_pol}")

## 1.2 Предварительные оценки

In [ ]:
k_avg = k0;  cp_avg = cp0
for _iter in range(20):
    eta_k_star_est = (pi_k_star**((k_avg-1)/k_avg) - 1) / (pi_k_star**((k_avg-1)/(k_avg*eta_pol)) - 1)
    T_k_star = T0_star * (1 + (pi_k_star**((k_avg-1)/k_avg) - 1) / eta_k_star_est)
    k_avg_new = k_air(0.5*(T0_star + T_k_star), Rg)
    cp_avg_new = cp_air(0.5*(T0_star + T_k_star), Rg)
    if abs(k_avg_new - k_avg) < 1e-6:
        k_avg, cp_avg = k_avg_new, cp_avg_new
        break
    k_avg, cp_avg = k_avg_new, cp_avg_new

k_vykh = k_air(T_k_star, Rg)
cp_vykh = cp_air(T_k_star, Rg)

print("=== eta*k через eta_pol ===")
print(f"  eta*k = (pi*k^((k-1)/k) - 1) / (pi*k^((k-1)/(k*eta_pol)) - 1)")
print(f"        = ({pi_k_star}^(({k_avg:.4f}-1)/{k_avg:.4f}) - 1) / ({pi_k_star}^(({k_avg:.4f}-1)/({k_avg:.4f}*{eta_pol})) - 1)")
print(f"        = {eta_k_star_est:.4f}")
print()
print("=== Формула (7): T*2 ===")
pifrac = pi_k_star**((k_avg-1)/k_avg) - 1
print(f"  T*2 = T*1 * (1 + (pi*k^((k-1)/k) - 1) / eta*k)")
print(f"      = {T0_star} * (1 + ({pifrac:.4f}) / {eta_k_star_est:.4f})")
print(f"      = {T0_star} * {1 + pifrac/eta_k_star_est:.4f}")
print(f"      = {T_k_star:.1f} К")
print()
print(f"  k_avg  = {k_avg:.4f}  (при T_avg = {0.5*(T0_star+T_k_star):.1f} К)")
print(f"  cp_avg = {cp_avg:.1f} Дж/(кг*К)")
print(f"  k_vykh = {k_vykh:.4f},  cp_vykh = {cp_vykh:.1f}")

## Формулы (1)-(21)

In [ ]:
C_vkh = 0.45 * 340;  C_vykh = 0.36 * 340

# (10) a_cr_vkh
a_cr_vkh = a_cr_func(T0_star, k=k0, Rg=Rg)
print("=== (10) a_кр_вх = sqrt(2k/(k+1) * Rg * T*1) ===")
print(f"  k0 = {k0:.4f},  Rg = {Rg},  T*1 = {T0_star}")
print(f"  = sqrt(2*{k0:.4f}/({k0:.4f}+1) * {Rg} * {T0_star})")
print(f"  = {a_cr_vkh:.1f} м/с")

# (11) a_cr_vykh
a_cr_vykh = a_cr_func(T_k_star, k=k_vykh, Rg=Rg)
print()
print("=== (11) a_кр_вых = sqrt(2k/(k+1) * Rg * T*2) ===")
print(f"  k_vykh = {k_vykh:.4f},  T*2 = {T_k_star:.1f}")
print(f"  = sqrt(2*{k_vykh:.4f}/({k_vykh:.4f}+1) * {Rg} * {T_k_star:.1f})")
print(f"  = {a_cr_vykh:.1f} м/с")

# (12) lam_vkh
lam_vkh = C_vkh / a_cr_vkh
print()
print("=== (12) lam_вх = c_вх / a_кр_вх ===")
print(f"  c_вх = {C_vkh:.1f},  a_кр_вх = {a_cr_vkh:.1f}")
print(f"  = {C_vkh:.1f} / {a_cr_vkh:.1f} = {lam_vkh:.4f}")

# (13) lam_vykh
lam_vykh = C_vykh / a_cr_vykh
print()
print("=== (13) lam_вых = c_вых / a_кр_вых ===")
print(f"  c_вых = {C_vykh:.1f},  a_кр_вых = {a_cr_vykh:.1f}")
print(f"  = {C_vykh:.1f} / {a_cr_vykh:.1f} = {lam_vykh:.4f}")

# (14) sigma_vkh
ev = eps_lam(lam_vkh, k0)
sigma_vkh = 1.0 / (1.0 + zeta_vkh * k0/(k0+1) * ev * lam_vkh**2)
print()
print("=== (14) sigma_вх = 1 / (1 + zeta_вх * eps(lam_вх) * k/(k+1) * lam_вх^2) ===")
print(f"  zeta_вх = {zeta_vkh},  eps(lam_вх) = {ev:.4f},  k0 = {k0:.4f},  lam_вх = {lam_vkh:.4f}")
print(f"  = 1 / (1 + {zeta_vkh} * {ev:.4f} * {k0:.4f}/{k0+1:.4f} * {lam_vkh:.4f}^2)")
print(f"  = 1 / (1 + {zeta_vkh * ev * k0/(k0+1) * lam_vkh**2:.6f})")
print(f"  = {sigma_vkh:.4f}")

# (15) sigma_vykh
ev2 = eps_lam(lam_vykh, k_vykh)
sigma_vykh = 1.0 - zeta_vykh * k_vykh/(k_vykh+1) * ev2 * lam_vykh**2
print()
print("=== (15) sigma_вых = 1 - zeta_вых * eps(lam_вых) * k/(k+1) * lam_вых^2 ===")
print(f"  zeta_вых = {zeta_vykh},  eps(lam_вых) = {ev2:.4f},  k_vykh = {k_vykh:.4f},  lam_вых = {lam_vykh:.4f}")
print(f"  = 1 - {zeta_vykh} * {ev2:.4f} * {k_vykh:.4f}/{k_vykh+1:.4f} * {lam_vykh:.4f}^2")
print(f"  = 1 - {zeta_vykh * ev2 * k_vykh/(k_vykh+1) * lam_vykh**2:.6f}")
print(f"  = {sigma_vykh:.4f}")

# (16) P1_star
P1_star = sigma_vkh * P0_star
print()
print("=== (16) P*1 = sigma_вх * P*a ===")
print(f"  sigma_вх = {sigma_vkh:.4f},  P*a = {P0_star:.0f}")
print(f"  = {sigma_vkh:.4f} * {P0_star:.0f} = {P1_star:.0f} Па")

# (17) P2_star
P_k_out = pi_k_star * P0_star
P2_star = P_k_out / sigma_vykh
print()
print("=== (17) P*2 = pi*k * P*a / sigma_вых ===")
print(f"  pi*k = {pi_k_star},  P*a = {P0_star:.0f},  sigma_вых = {sigma_vykh:.4f}")
print(f"  = {pi_k_star} * {P0_star:.0f} / {sigma_vykh:.4f} = {P2_star:.0f} Па")

# (18) pi_la
pi_la = P2_star / P1_star
print()
print("=== (18) pi*_ла = P*2 / P*1 ===")
print(f"  P*2 = {P2_star:.0f},  P*1 = {P1_star:.0f}")
print(f"  = {P2_star:.0f} / {P1_star:.0f} = {pi_la:.3f}")

# (19) eta_la
C_coef = (pi_k_star**((k_avg-1)/k_avg) - 1) / (pi_la**((k_avg-1)/k_avg) - 1)
eta_la = eta_k_star_est / C_coef
print()
print("=== (19) eta*_ла = eta*k / C ===")
print(f"  C = (pi*k^((k-1)/k)-1) / (pi*la^((k-1)/k)-1)")
print(f"    = ({pi_k_star**((k_avg-1)/k_avg)-1:.4f}) / ({pi_la**((k_avg-1)/k_avg)-1:.4f}) = {C_coef:.4f}")
print(f"  eta*k = {eta_k_star_est:.4f}")
print(f"  eta*_ла = {eta_k_star_est:.4f} / {C_coef:.4f} = {eta_la:.4f}")

# (3) H_ad_k
H_ad_k = cp_avg * T0_star * (pi_k_star**((k_avg-1)/k_avg) - 1)
print()
print("=== (3) H*_ад_к = cp_avg * T*0 * (pi*k^((k-1)/k) - 1) ===")
print(f"  cp_avg = {cp_avg:.1f},  T*0 = {T0_star},  pi*k = {pi_k_star}")
print(f"  pi*k^((k-1)/k) - 1 = {pi_k_star**((k_avg-1)/k_avg) - 1:.4f}")
print(f"  = {cp_avg:.1f} * {T0_star} * {pi_k_star**((k_avg-1)/k_avg)-1:.4f} = {H_ad_k:.0f} Дж/кг")

# (4) H_tk
H_tk = H_ad_k / eta_k_star_est
print()
print("=== (4) H_тк = H*_ад_к / eta*k ===")
print(f"  H*_ад_к = {H_ad_k:.0f},  eta*k = {eta_k_star_est:.4f}")
print(f"  = {H_ad_k:.0f} / {eta_k_star_est:.4f} = {H_tk:.0f} Дж/кг")

# (20) F_vkh
beta_vkh_val = beta_coef(k0)
qv = q_lam(lam_vkh, k0)
F_vkh_area = G * np.sqrt(Rg * T0_star) / (beta_vkh_val * P1_star * qv)
print()
print("=== (20) F_вх = G * sqrt(Rg*T*1) / (beta*P*1*q(lam_вх)) ===")
print(f"  G = {G},  Rg = {Rg},  T*1 = {T0_star}")
print(f"  beta(k0) = {beta_vkh_val:.4f},  P*1 = {P1_star:.0f},  q(lam_вх) = {qv:.4f}")
print(f"  числитель = {G} * sqrt({Rg}*{T0_star}) = {G*np.sqrt(Rg*T0_star):.1f}")
print(f"  знаменатель = {beta_vkh_val:.4f} * {P1_star:.0f} * {qv:.4f} = {beta_vkh_val*P1_star*qv:.1f}")
print(f"  = {F_vkh_area:.4f} м2")

# (21) F_vykh
beta_vykh_val = beta_coef(k_vykh)
qv2 = q_lam(lam_vykh, k_vykh)
F_vykh_area = G * np.sqrt(Rg * T_k_star) / (beta_vykh_val * P2_star * qv2)
print()
print("=== (21) F_вых = G * sqrt(Rg*T*2) / (beta*P*2*q(lam_вых)) ===")
print(f"  G = {G},  Rg = {Rg},  T*2 = {T_k_star:.1f}")
print(f"  beta(k_vykh) = {beta_vykh_val:.4f},  P*2 = {P2_star:.0f},  q(lam_вых) = {qv2:.4f}")
print(f"  = {F_vykh_area:.4f} м2")

## Итерационный расчёт $U_k$ — формулы (22)-(27)

In [ ]:
HT_bar_1 = HT_bar_poly(ca_bar_1)

# (22)
rau_sr1 = np.sqrt((1.0 + d_bar_1**2) / 2.0)
print("=== (22) r_ср1 = sqrt((1 + d_bar_1^2) / 2) ===")
print(f"  d_bar_1 = {d_bar_1}")
print(f"  = sqrt((1 + {d_bar_1}^2) / 2) = sqrt({(1+d_bar_1**2)/2:.4f}) = {rau_sr1:.4f}")

# (23)
cu_bar1 = rau_sr1 * (1.0 - R_sr_base) - HT_bar_1 / (2.0 * rau_sr1)
print()
print("=== (23) cu_bar1 = r_ср1*(1-R) - HT_bar1/(2*r_ср1) ===")
print(f"  r_ср1 = {rau_sr1:.4f},  R = {R_sr_base},  HT_bar1 = {HT_bar_1:.4f}")
print(f"  = {rau_sr1:.4f}*(1-{R_sr_base}) - {HT_bar_1:.4f}/(2*{rau_sr1:.4f})")
print(f"  = {rau_sr1*(1-R_sr_base):.4f} - {HT_bar_1/(2*rau_sr1):.4f} = {cu_bar1:.4f}")

# Iterations
u_k = 300.0
print()
print("=== Итерации U_k (формулы 24-27) ===")
for it in range(200):
    # (24) c1
    c1_val = u_k * np.sqrt(ca_bar_1**2 + cu_bar1**2)
    # (25) lam1
    lam1_val = c1_val / a_cr_vkh
    # (26) rho1
    rho1_val = P1_star / (Rg * T0_star) * eps_lam(lam1_val, k0)
    # (27) u_k
    u_k_new = (4.0 * np.pi * G * (n_rpm/60.0)**2 / (ca_bar_1 * rho1_val * (1.0 - d_bar_1**2)))**(1.0/3.0)
    err = abs(u_k_new - u_k) / u_k * 100
    if it < 3:
        print(f"  Ит.{it+1}:")
        print(f"    (24) c1 = Uk*sqrt(ca^2+cu^2) = {u_k:.2f}*sqrt({ca_bar_1:.4f}^2+{cu_bar1:.4f}^2) = {c1_val:.2f}")
        print(f"    (25) lam1 = c1/a_cr = {c1_val:.2f}/{a_cr_vkh:.1f} = {lam1_val:.4f}")
        print(f"    (26) rho1 = P*1/(Rg*T*1)*eps(lam1) = {P1_star:.0f}/({Rg}*{T0_star})*{eps_lam(lam1_val,k0):.4f} = {rho1_val:.4f}")
        print(f"    (27) Uk = (4pi*G*(n/60)^2 / (ca*rho*(1-d^2)))^(1/3)")
        print(f"         = (4*pi*{G}*{n_rpm/60:.1f}^2 / ({ca_bar_1:.4f}*{rho1_val:.4f}*{1-d_bar_1**2:.4f}))^(1/3)")
        print(f"         = {u_k_new:.2f} м/с  (delta = {err:.3f}%)")
    if abs(u_k_new - u_k) < 0.001:
        u_k = u_k_new
        if it >= 3:
            print(f"  ... Сошлось на итерации {it+1}: Uk = {u_k:.2f} м/с")
        break
    u_k = 0.5 * u_k + 0.5 * u_k_new

## Геометрия — формулы (28)-(34)

In [ ]:
HT_bar_avg = 0.28
H_T_sr = HT_bar_avg * u_k**2

# (5)
print("=== (5) H_T_ср = HT_bar_avg * Uk^2 ===")
print(f"  HT_bar_avg = {HT_bar_avg},  Uk = {u_k:.2f}")
print(f"  = {HT_bar_avg} * {u_k:.2f}^2 = {H_T_sr:.0f} Дж/кг")

# (6)
Z_est = H_tk / H_T_sr
Z = int(np.ceil(Z_est));  Z = max(Z, 4)
print()
print("=== (6) Z = H_тк / H_T_ср ===")
print(f"  H_тк = {H_tk:.0f},  H_T_ср = {H_T_sr:.0f}")
print(f"  = {H_tk:.0f} / {H_T_sr:.0f} = {Z_est:.1f} -> Z = {Z}")

# (28)
D_k = 60.0 * u_k / (np.pi * n_rpm)
print()
print("=== (28) D_к1 = 60*Uk / (pi*n) ===")
print(f"  Uk = {u_k:.2f},  n = {n_rpm}")
print(f"  = 60*{u_k:.2f} / (pi*{n_rpm}) = {D_k:.4f} м = {D_k*1000:.1f} мм")

# (29)
D_vt_1 = d_bar_1 * D_k
print()
print("=== (29) D_вт1 = d_bar_1 * D_к1 ===")
print(f"  d_bar_1 = {d_bar_1},  D_к1 = {D_k:.4f}")
print(f"  = {d_bar_1} * {D_k:.4f} = {D_vt_1:.4f} м = {D_vt_1*1000:.1f} мм")

# (30)
D_sr_1 = rau_sr1 * D_k
print()
print("=== (30) D_ср1 = r_ср1 * D_к1 ===")
print(f"  r_ср1 = {rau_sr1:.4f},  D_к1 = {D_k:.4f}")
print(f"  = {rau_sr1:.4f} * {D_k:.4f} = {D_sr_1:.4f} м = {D_sr_1*1000:.1f} мм")

# (31)
h_1 = 0.5 * D_k * (1.0 - d_bar_1)
print()
print("=== (31) h_1 = 0.5 * D_к1 * (1 - d_bar_1) ===")
print(f"  D_к1 = {D_k:.4f},  d_bar_1 = {d_bar_1}")
print(f"  = 0.5 * {D_k:.4f} * (1 - {d_bar_1}) = 0.5 * {D_k:.4f} * {1-d_bar_1:.4f} = {h_1:.4f} м = {h_1*1000:.1f} мм")

# (32)
D_vt_n_sq = D_k**2 - 4.0 * F_vykh_area / np.pi
D_vt_n = np.sqrt(max(D_vt_n_sq, 0))
print()
print("=== (32) D_вт_п = sqrt(D_к^2 - 4*F_вых/pi) ===")
print(f"  D_к = {D_k:.4f},  F_вых = {F_vykh_area:.4f}")
print(f"  = sqrt({D_k:.4f}^2 - 4*{F_vykh_area:.4f}/pi) = sqrt({D_vt_n_sq:.6f}) = {D_vt_n:.4f} м")

# (33)
d_bar_n = D_vt_n / D_k
print()
print("=== (33) d_bar_п = D_вт_п / D_к ===")
print(f"  = {D_vt_n:.4f} / {D_k:.4f} = {d_bar_n:.4f}")

# (34)
h_n = 0.5 * D_k * (1.0 - d_bar_n)
print()
print("=== (34) h_п = 0.5 * D_к * (1 - d_bar_п) ===")
print(f"  = 0.5 * {D_k:.4f} * (1 - {d_bar_n:.4f}) = {h_n:.4f} м = {h_n*1000:.1f} мм")

H_MIN_MM = 15.0;  D_BAR_MAX = 0.92
if d_bar_n >= D_BAR_MAX or h_n * 1000 < H_MIN_MM:
    d_bar_switch = min(D_BAR_MAX - 0.02, d_bar_n - 0.02)
    d_bar_switch = max(d_bar_switch, d_bar_1 + 0.05)
    D_vt_switch = d_bar_switch * D_k
    D_k_exit = np.sqrt(D_vt_switch**2 + 4.0 * F_vykh_area / np.pi)
    d_bar_n = D_vt_switch / D_k_exit
    h_n = (D_k_exit - D_vt_switch) / 2.0
    print(f"\n  -> Переключение Dk=const -> Dvt=const (d_bar_n>={D_BAR_MAX} или h<{H_MIN_MM}мм)")
    print(f"     d_bar_n (скорректированный) = {d_bar_n:.4f},  h_n = {h_n*1000:.1f} мм")

## 2.1 Распределение параметров по ступеням

In [ ]:
stages = np.arange(1, Z + 1)
ca_bar_last = max(0.36, ca_bar_1 - 0.018 * (Z - 1))
ca_bar_dist = ca_bar_1 - (ca_bar_1 - ca_bar_last) * ((stages - 1) / max(Z - 1, 1))**0.75
eta_dist = np.array([eta_stage_poly(c) for c in ca_bar_dist])
eta_corr = np.ones(Z)
eta_corr[0] = 0.975
if Z > 1: eta_corr[1] = 0.988
if Z > 2: eta_corr[-1] = 0.975
if Z > 3: eta_corr[-2] = 0.988
eta_dist *= eta_corr
HT_bar_dist = np.array([HT_bar_poly(c) for c in ca_bar_dist])
ht_corr = np.ones(Z)
ht_corr[0] = 0.88
if Z > 1: ht_corr[1] = 0.94
if Z > 2: ht_corr[-1] = 0.90
if Z > 3: ht_corr[-2] = 0.95
HT_bar_dist *= ht_corr
R_dist = np.full(Z, 0.5)
if Z > 3: R_dist[-1] = 0.55; R_dist[-2] = 0.52
K_H_dist = np.linspace(0.99, max(0.94, 0.99 - 0.005 * (Z - 1)), Z)
delta_Ca_dist = np.linspace(6, 10, Z)
H_tk_dist = np.sum(HT_bar_dist) * u_k**2
scale_HT = H_tk / H_tk_dist
HT_bar_dist *= scale_HT
print(f"Масштаб HT_bar: {scale_HT:.4f}")
print(f"{'Ступ':>5} {'ca_bar':>8} {'HT_bar':>8} {'eta*':>8} {'R':>6} {'K_H':>6}")
for i in range(Z):
    print(f"{i+1:>5d} {ca_bar_dist[i]:>8.4f} {HT_bar_dist[i]:>8.4f} {eta_dist[i]:>8.4f} {R_dist[i]:>6.3f} {K_H_dist[i]:>6.3f}")

## 2.2 Поступенчатый расчёт — формулы (35)-(88)

Подробный вывод для **1-й ступени**, затем автоматически для остальных.

In [ ]:
T1s=np.zeros(Z+1); P1s=np.zeros(Z+1); T1s[0]=T0_star; P1s[0]=P1_star
HT_dim=np.zeros(Z); Lz=np.zeros(Z); Had=np.zeros(Z); dT=np.zeros(Z)
pi_st=np.zeros(Z); acr1=np.zeros(Z); acr3=np.zeros(Z); rcp1=np.zeros(Z)
cu1_bar=np.zeros(Z); alpha1=np.zeros(Z); lam1=np.zeros(Z); F1=np.zeros(Z)
d1_bar=np.zeros(Z); F3=np.zeros(Z); d3_bar=np.zeros(Z); rcp3=np.zeros(Z)
rcp2=np.zeros(Z); cu2_bar=np.zeros(Z); cu3_bar_arr=np.zeros(Z)
beta1=np.zeros(Z); beta2=np.zeros(Z); alpha2=np.zeros(Z); alpha3=np.zeros(Z)
eps_RK=np.zeros(Z); eps_NA=np.zeros(Z); w1=np.zeros(Z); c2=np.zeros(Z)
Mw1_sr=np.zeros(Z); Mc2_sr=np.zeros(Z)
ca1_dim=np.zeros(Z); ca2_dim=np.zeros(Z); ca2_bar=np.zeros(Z); h_blade=np.zeros(Z)
Dk_arr=np.zeros(Z+1); Dvt_arr=np.zeros(Z+1); k_st=np.zeros(Z); cp_st=np.zeros(Z)
d2_bar_arr=np.zeros(Z)

HT_bar_dist_orig = HT_bar_dist.copy()

def run_stages(verbose_stage=-1):
    global T1s, P1s, HT_dim, Lz, Had, dT, pi_st, acr1, acr3, rcp1
    global cu1_bar, alpha1, lam1, F1, d1_bar, F3, d3_bar, rcp3, rcp2
    global cu2_bar, cu3_bar_arr, beta1, beta2, alpha2, alpha3
    global eps_RK, eps_NA, w1, c2, Mw1_sr, Mc2_sr
    global ca1_dim, ca2_dim, ca2_bar, h_blade, Dk_arr, Dvt_arr, k_st, cp_st, d2_bar_arr
    
    T1s[0]=T0_star; P1s[0]=P1_star
    dvt_c=False; D_vt_f=0.0; i_sw=-1
    d1_bar[0]=d_bar_1; Dk_arr[0]=D_k; Dvt_arr[0]=d_bar_1*D_k
    
    for i in range(Z):
        v = (i == verbose_stage)
        k_i=k_air(T1s[i],Rg); cp_i=cp_air(T1s[i],Rg)
        k_st[i]=k_i; cp_st[i]=cp_i; beta_i=beta_coef(k_i)
        
        if v: print(f"\n{'='*70}\n  СТУПЕНЬ {i+1}: подробный расчёт\n{'='*70}")
        if v: print(f"  k({T1s[i]:.1f} К) = {k_i:.4f},  cp = {cp_i:.1f}")
        
        # (35)
        HT_dim[i] = HT_bar_dist[i]*u_k**2
        if v:
            print(f"\n  (35) H_T = HT_bar * Uk^2 = {HT_bar_dist[i]:.4f} * {u_k:.2f}^2 = {HT_dim[i]:.0f} Дж/кг")
        
        # (36)
        Lz[i] = K_H_dist[i]*HT_dim[i]
        if v:
            print(f"  (36) L_z = K_H * H_T = {K_H_dist[i]:.3f} * {HT_dim[i]:.0f} = {Lz[i]:.0f} Дж/кг")
        
        # (37)
        Had[i] = Lz[i]*eta_dist[i]
        if v:
            print(f"  (37) H_ад = L_z * eta* = {Lz[i]:.0f} * {eta_dist[i]:.4f} = {Had[i]:.0f} Дж/кг")
        
        # (38)
        dT[i] = Lz[i]/cp_i
        if v:
            print(f"  (38) dT* = L_z / cp = {Lz[i]:.0f} / {cp_i:.1f} = {dT[i]:.2f} К")
        
        # (39)
        T1s[i+1] = T1s[i]+dT[i]
        if v:
            print(f"  (39) T*3 = T*1 + dT* = {T1s[i]:.1f} + {dT[i]:.2f} = {T1s[i+1]:.1f} К")
        
        # (40)
        pi_st[i] = (1.0+Had[i]/(cp_i*T1s[i]))**(k_i/(k_i-1.0))
        if v:
            hct = Had[i]/(cp_i*T1s[i])
            print(f"  (40) pi* = (1 + H_ад/(cp*T*1))^(k/(k-1))")
            print(f"       = (1 + {Had[i]:.0f}/({cp_i:.1f}*{T1s[i]:.1f}))^({k_i:.4f}/{k_i-1:.4f})")
            print(f"       = (1 + {hct:.6f})^{k_i/(k_i-1):.4f} = {pi_st[i]:.4f}")
        
        # (41)
        P1s[i+1] = P1s[i]*pi_st[i]
        if v:
            print(f"  (41) P*3 = P*1 * pi* = {P1s[i]:.0f} * {pi_st[i]:.4f} = {P1s[i+1]:.0f} Па")
        
        k_i3=k_air(T1s[i+1],Rg); beta_i3=beta_coef(k_i3)
        # (42, 43)
        acr1[i]=a_cr_func(T1s[i],k=k_i,Rg=Rg)
        acr3[i]=a_cr_func(T1s[i+1],k=k_i3,Rg=Rg)
        if v:
            print(f"  (42) a_кр1 = sqrt(2k/(k+1)*Rg*T*1) = sqrt(2*{k_i:.4f}/{k_i+1:.4f}*{Rg}*{T1s[i]:.1f}) = {acr1[i]:.1f}")
            print(f"  (43) a_кр3 = sqrt(2k/(k+1)*Rg*T*3) = sqrt(2*{k_i3:.4f}/{k_i3+1:.4f}*{Rg}*{T1s[i+1]:.1f}) = {acr3[i]:.1f}")
        
        # (44)
        rcp1[i]=np.sqrt((1.0+d1_bar[i]**2)/2.0)
        if v:
            print(f"  (44) r_ср1 = sqrt((1+d1^2)/2) = sqrt((1+{d1_bar[i]:.4f}^2)/2) = {rcp1[i]:.4f}")
        
        # (45)
        cu1_bar[i]=rcp1[i]*(1.0-R_dist[i])-HT_bar_dist[i]/(2.0*rcp1[i])
        if v:
            print(f"  (45) cu1_bar = r_ср1*(1-R) - HT_bar/(2*r_ср1)")
            print(f"       = {rcp1[i]:.4f}*(1-{R_dist[i]:.2f}) - {HT_bar_dist[i]:.4f}/(2*{rcp1[i]:.4f})")
            print(f"       = {rcp1[i]*(1-R_dist[i]):.4f} - {HT_bar_dist[i]/(2*rcp1[i]):.4f} = {cu1_bar[i]:.4f}")
        
        u_kl=np.pi*Dk_arr[i]*n_rpm/60.0
        ca1_dim[i]=ca_bar_dist[i]*u_kl
        ca2_dim[i]=ca1_dim[i]-delta_Ca_dist[i]
        ca2_bar[i]=ca2_dim[i]/u_kl
        
        if v:
            print(f"  (47) ca1 = ca_bar*Uk = {ca_bar_dist[i]:.4f}*{u_kl:.2f} = {ca1_dim[i]:.1f} м/с")
            print(f"  (50) ca2 = ca1 - dCa = {ca1_dim[i]:.1f} - {delta_Ca_dist[i]:.1f} = {ca2_dim[i]:.1f} м/с")
        
        # (46)
        alpha1[i]=np.arctan2(ca_bar_dist[i],cu1_bar[i]) if abs(cu1_bar[i])>1e-10 else np.pi/2
        if v:
            print(f"  (46) alpha1 = atan(ca_bar/cu_bar) = atan({ca_bar_dist[i]:.4f}/{cu1_bar[i]:.4f}) = {np.degrees(alpha1[i]):.1f} град")
        
        # (48)
        lam1[i]=ca1_dim[i]/(np.sin(alpha1[i])*acr1[i])
        if v:
            print(f"  (48) lam1 = ca1/(sin(alpha1)*a_кр1) = {ca1_dim[i]:.1f}/(sin({np.degrees(alpha1[i]):.1f})*{acr1[i]:.1f})")
            print(f"       = {ca1_dim[i]:.1f}/({np.sin(alpha1[i]):.4f}*{acr1[i]:.1f}) = {lam1[i]:.4f}")
        
        # (49)
        q1v=q_lam(lam1[i],k_i)
        F1[i]=G*np.sqrt(Rg*T1s[i])/(beta_i*P1s[i]*q1v*np.sin(alpha1[i])) if q1v>0 else np.pi/4*Dk_arr[i]**2*(1-d1_bar[i]**2)
        if v:
            print(f"  (49) F1 = G*sqrt(Rg*T*1) / (beta*P*1*q(lam1)*sin(alpha1))")
            print(f"       q(lam1) = {q1v:.4f},  beta = {beta_i:.4f}")
            print(f"       = {G}*{np.sqrt(Rg*T1s[i]):.1f} / ({beta_i:.4f}*{P1s[i]:.0f}*{q1v:.4f}*{np.sin(alpha1[i]):.4f})")
            print(f"       = {F1[i]:.4f} м2")
        
        if i>0:
            if not dvt_c:
                d1_bar[i]=np.sqrt(max(1.0-4.0*F1[i]/(np.pi*D_k**2),0.01))
                Dk_arr[i]=D_k; Dvt_arr[i]=d1_bar[i]*D_k
            else:
                Dk_arr[i]=np.sqrt(D_vt_f**2+4.0*F1[i]/np.pi)
                Dvt_arr[i]=D_vt_f; d1_bar[i]=D_vt_f/Dk_arr[i]
                u_kl=np.pi*Dk_arr[i]*n_rpm/60.0
                ca1_dim[i]=ca_bar_dist[i]*u_kl; ca2_dim[i]=ca1_dim[i]-delta_Ca_dist[i]; ca2_bar[i]=ca2_dim[i]/u_kl
            rcp1[i]=np.sqrt((1.0+d1_bar[i]**2)/2.0)
            cu1_bar[i]=rcp1[i]*(1.0-R_dist[i])-HT_bar_dist[i]/(2.0*rcp1[i])
            alpha1[i]=np.arctan2(ca_bar_dist[i],cu1_bar[i]) if abs(cu1_bar[i])>1e-10 else np.pi/2
            lam1[i]=ca1_dim[i]/(np.sin(alpha1[i])*acr1[i])
        
        a3a=alpha1[i] if i<Z-1 else np.pi/2
        l3a=ca2_dim[i]/(np.sin(a3a)*acr3[i])
        q3a=q_lam(l3a,k_i3)
        F3a=G*np.sqrt(Rg*T1s[i+1])/(beta_i3*P1s[i+1]*q3a*np.sin(a3a)) if q3a>0 else F1[i]*0.95
        if not dvt_c: d3a=np.sqrt(max(1.0-4.0*F3a/(np.pi*D_k**2),0.01))
        else: d3a=D_vt_f/np.sqrt(D_vt_f**2+4.0*F3a/np.pi)
        rcp3a=np.sqrt((1.0+d3a**2)/2.0)
        if i<Z-1:
            cu3b=rcp3a*(1.0-R_dist[i+1])-HT_bar_dist[i+1]/(2.0*rcp3a)
            alpha3[i]=np.arctan2(ca_bar_dist[i+1],cu3b) if abs(cu3b)>1e-10 else np.pi/2
        else: alpha3[i]=np.pi/2
        l3=ca2_dim[i]/(np.sin(alpha3[i])*acr3[i])
        q3v=q_lam(l3,k_i3)
        F3[i]=G*np.sqrt(Rg*T1s[i+1])/(beta_i3*P1s[i+1]*q3v*np.sin(alpha3[i])) if q3v>0 else F3a
        if not dvt_c: d3_bar[i]=np.sqrt(max(1.0-4.0*F3[i]/(np.pi*D_k**2),0.01)); Dk3f=D_k
        else: Dk3f=np.sqrt(D_vt_f**2+4.0*F3[i]/np.pi); d3_bar[i]=D_vt_f/Dk3f
        h3=(Dk3f-d3_bar[i]*Dk3f)/2.0
        if not dvt_c and (h3*1000<H_MIN_MM or d3_bar[i]>=D_BAR_MAX):
            dvt_c=True; i_sw=i; D_vt_f=Dvt_arr[i]
            Dk3f=np.sqrt(D_vt_f**2+4.0*F3[i]/np.pi); d3_bar[i]=D_vt_f/Dk3f
        if not dvt_c: Dk_arr[i+1]=D_k; Dvt_arr[i+1]=d3_bar[i]*D_k
        else: Dk_arr[i+1]=Dk3f; Dvt_arr[i+1]=D_vt_f
        if i<Z-1: d1_bar[i+1]=d3_bar[i]
        
        rcp3[i]=np.sqrt((1.0+d3_bar[i]**2)/2.0)
        rcp2[i]=0.5*(rcp1[i]+rcp3[i])
        d2_bar_arr[i]=np.sqrt(max(2.0*rcp2[i]**2-1.0,0.01))
        cu2_bar[i]=(HT_bar_dist[i]+cu1_bar[i]*rcp1[i])/rcp2[i]
        if i<Z-1: cu3_bar_arr[i]=rcp3[i]*(1.0-R_dist[i+1])-HT_bar_dist[i+1]/(2.0*rcp3[i])
        beta1[i]=np.arctan2(ca_bar_dist[i],rcp1[i]-cu1_bar[i])
        beta2[i]=np.arctan2(ca2_bar[i],rcp2[i]-cu2_bar[i])
        alpha2[i]=np.arctan2(ca2_bar[i],cu2_bar[i])
        eps_RK[i]=beta2[i]-beta1[i]; eps_NA[i]=alpha3[i]-alpha2[i]
        w1[i]=ca1_dim[i]/np.sin(beta1[i]) if np.sin(beta1[i])>0.01 else ca1_dim[i]
        c2[i]=ca2_dim[i]/np.sin(alpha2[i]) if np.sin(alpha2[i])>0.01 else ca2_dim[i]
        T1_stat=T1s[i]*tau_lam(lam1[i],k_i)
        if T1_stat>0: Mw1_sr[i]=w1[i]/np.sqrt(k_i*Rg*T1_stat)
        lam_c2=c2[i]/acr3[i]; T2_stat=T1s[i+1]*tau_lam(lam_c2,k_i3)
        if T2_stat>0: Mc2_sr[i]=c2[i]/np.sqrt(k_i3*Rg*T2_stat)
        h_blade[i]=(Dk_arr[i]-Dvt_arr[i])/2.0
        
        if v:
            print(f"\n  --- Выходные параметры ступени {i+1} ---")
            print(f"  (55) r_ср3 = {rcp3[i]:.4f}")
            print(f"  (67) r_ср2 = (r_ср1+r_ср3)/2 = ({rcp1[i]:.4f}+{rcp3[i]:.4f})/2 = {rcp2[i]:.4f}")
            print(f"  (69) d2_bar = sqrt(2*r_ср2^2 - 1) = sqrt(2*{rcp2[i]:.4f}^2-1) = {d2_bar_arr[i]:.4f}")
            print(f"  (71) cu2_bar = (HT_bar + cu1*r_ср1)/r_ср2 = ({HT_bar_dist[i]:.4f}+{cu1_bar[i]:.4f}*{rcp1[i]:.4f})/{rcp2[i]:.4f} = {cu2_bar[i]:.4f}")
            print(f"  (73) beta1 = atan(ca_bar/(r_ср1-cu1)) = atan({ca_bar_dist[i]:.4f}/({rcp1[i]:.4f}-{cu1_bar[i]:.4f})) = {np.degrees(beta1[i]):.1f} град")
            print(f"  (74) beta2 = atan(ca2_bar/(r_ср2-cu2)) = atan({ca2_bar[i]:.4f}/({rcp2[i]:.4f}-{cu2_bar[i]:.4f})) = {np.degrees(beta2[i]):.1f} град")
            print(f"  (75) alpha2 = atan(ca2_bar/cu2_bar) = atan({ca2_bar[i]:.4f}/{cu2_bar[i]:.4f}) = {np.degrees(alpha2[i]):.1f} град")
            print(f"  (76) eps_РК = beta2-beta1 = {np.degrees(beta2[i]):.1f}-{np.degrees(beta1[i]):.1f} = {np.degrees(eps_RK[i]):.1f} град")
            print(f"  (77) eps_НА = alpha3-alpha2 = {np.degrees(alpha3[i]):.1f}-{np.degrees(alpha2[i]):.1f} = {np.degrees(eps_NA[i]):.1f} град")
            print(f"  (79) w1 = ca1/sin(beta1) = {ca1_dim[i]:.1f}/sin({np.degrees(beta1[i]):.1f}) = {w1[i]:.1f} м/с")
            print(f"  (80) c2 = ca2/sin(alpha2) = {ca2_dim[i]:.1f}/sin({np.degrees(alpha2[i]):.1f}) = {c2[i]:.1f} м/с")
            print(f"  (85) Mw1 = w1/sqrt(k*Rg*T1_stat) = {w1[i]:.1f}/sqrt({k_i:.4f}*{Rg}*{T1_stat:.1f}) = {Mw1_sr[i]:.4f}")
            print(f"  (86) Mc2 = c2/sqrt(k*Rg*T2_stat) = {c2[i]:.1f}/sqrt({k_i3:.4f}*{Rg}*{T2_stat:.1f}) = {Mc2_sr[i]:.4f}")
            print(f"  (87) h_РК = (Dk-Dvt)/2 = ({Dk_arr[i]*1000:.1f}-{Dvt_arr[i]*1000:.1f})/2 = {h_blade[i]*1000:.1f} мм")
            print(f"  d3_bar = {d3_bar[i]:.4f},  F3 = {F3[i]:.4f} м2")
    
    return i_sw, D_vt_f, dvt_c

# Итерации коррекции
for n_iter in range(20):
    i_sw, D_vt_f, dvt_c = run_stages(verbose_stage=0 if n_iter==0 else -1)
    pi_la_calc=P1s[Z]/P1s[0]
    pi_k_calc=pi_la_calc*sigma_vkh*sigma_vykh
    delta_pi=(pi_k_calc-pi_k_star)/pi_k_star*100
    if abs(delta_pi)<1.0:
        print(f"\nИтерация {n_iter+1}: dpi* = {delta_pi:+.3f}% -- СОШЛОСЬ")
        break
    corr=np.log(pi_la)/np.log(pi_la_calc)
    HT_bar_dist*=corr
    print(f"Итерация {n_iter+1}: dpi* = {delta_pi:+.2f}%, корр = {corr:.4f}")

i_switch=i_sw; D_vt_frozen=D_vt_f; dvt_const_mode=dvt_c
print(f"Финал: dpi* = {delta_pi:+.3f}%")
if i_switch>=0: print(f"Переключение Dk->Dvt на ступени {i_switch+1}")


### Таблица результатов

In [ ]:
print("=" * 120)
hdr = f"{'Параметр':<16} {'Разм.':<8}"
for i in range(Z): hdr += f"{'Ст.'+str(i+1):>9}"
print(hdr); print("-"*120)
def row(n,u,a,f=".1f"):
    s=f"{n:<16} {u:<8}"
    for v in a: s+=f"{v:>9{f}}"
    print(s)
row("HT_bar","-",HT_bar_dist,".4f"); row("H_T","Дж/кг",HT_dim,".0f")
row("L_z","Дж/кг",Lz,".0f"); row("H_ад","Дж/кг",Had,".0f")
row("dT*","К",dT,".2f"); row("T*_1","К",T1s[:Z],".1f"); row("T*_3","К",T1s[1:Z+1],".1f")
row("pi*_ст","-",pi_st,".4f")
row("P*_1","кПа",P1s[:Z]/1000,".2f"); row("P*_3","кПа",P1s[1:Z+1]/1000,".2f")
row("d_bar_1","-",d1_bar,".4f"); row("d_bar_3","-",d3_bar,".4f")
row("h_лоп","мм",h_blade*1000,".1f")
row("alpha1","°",np.degrees(alpha1),".1f"); row("alpha2","°",np.degrees(alpha2),".1f"); row("alpha3","°",np.degrees(alpha3),".1f")
row("beta1","°",np.degrees(beta1),".1f"); row("beta2","°",np.degrees(beta2),".1f")
row("eps_РК","°",np.degrees(eps_RK),".1f"); row("eps_НА","°",np.degrees(eps_NA),".1f")
row("w1","м/с",w1,".1f"); row("c2","м/с",c2,".1f")
row("Mw1_ср","-",Mw1_sr,".3f"); row("Mc2_ср","-",Mc2_sr,".3f")

## 2.3 Уточнение — формулы (89)-(100)

In [ ]:
pi_la_calc=P1s[Z]/P1s[0]
k_avg_ver=k_air(0.5*(T1s[0]+T1s[Z]),Rg)
eta_la_calc=T0_star*(pi_la_calc**((k_avg_ver-1)/k_avg_ver)-1)/(T1s[Z]-T1s[0])
pi_k_calc=pi_la_calc*sigma_vkh*sigma_vykh
eta_k_calc=T0_star*(pi_k_calc**((k_avg_ver-1)/k_avg_ver)-1)/(T1s[Z]-T0_star)
N_k=G*np.sum(Lz)/1000
delta_pi=(pi_k_calc-pi_k_star)/pi_k_star*100

print("=== (89) pi*_ла = P*3_Z / P*1_1 ===")
print(f"  = {P1s[Z]:.0f} / {P1s[0]:.0f} = {pi_la_calc:.3f}")
print()
print("=== (90) eta*_ла = T*1*(pi*_ла^((k-1)/k)-1) / (T*3_Z - T*1) ===")
pf = pi_la_calc**((k_avg_ver-1)/k_avg_ver)-1
print(f"  T*1 = {T0_star},  pi*_ла = {pi_la_calc:.3f},  k_avg = {k_avg_ver:.4f}")
print(f"  pi*_ла^((k-1)/k)-1 = {pf:.4f}")
print(f"  T*3_Z - T*1 = {T1s[Z]:.1f} - {T1s[0]:.1f} = {T1s[Z]-T1s[0]:.1f}")
print(f"  = {T0_star}*{pf:.4f}/{T1s[Z]-T1s[0]:.1f} = {eta_la_calc:.4f}")
print()
print("=== (99) pi*_к = pi*_ла * sigma_вх * sigma_вых ===")
print(f"  = {pi_la_calc:.3f} * {sigma_vkh:.4f} * {sigma_vykh:.4f} = {pi_k_calc:.3f}")
print()
print("=== (100) eta*_к = T*1*(pi*_к^((k-1)/k)-1) / (T*3_Z - T*1) ===")
pf2 = pi_k_calc**((k_avg_ver-1)/k_avg_ver)-1
print(f"  pi*_к^((k-1)/k)-1 = {pf2:.4f}")
print(f"  = {T0_star}*{pf2:.4f}/{T1s[Z]-T0_star:.1f} = {eta_k_calc:.4f}")
print()
print(f"  N_к = G*sum(Lz) = {G}*{np.sum(Lz):.0f} = {N_k:.0f} кВт = {N_k/1000:.1f} МВт")
print(f"  Невязка: dpi* = {delta_pi:+.2f}%")

## 3. Три сечения — формулы (101)-(150)

Подробно для 1-й ступени.

In [ ]:
omega=2.0*np.pi*n_rpm/60.0
c1a_r=np.zeros((Z,3));c1u_r=np.zeros((Z,3));al1_r=np.zeros((Z,3));c1_r=np.zeros((Z,3))
w1a_r=np.zeros((Z,3));w1u_r=np.zeros((Z,3));bt1_r=np.zeros((Z,3));w1_r=np.zeros((Z,3))
c2a_r=np.zeros((Z,3));c2u_r=np.zeros((Z,3));al2_r=np.zeros((Z,3));c2_r=np.zeros((Z,3))
w2a_r=np.zeros((Z,3));w2u_r=np.zeros((Z,3));bt2_r=np.zeros((Z,3));w2_r=np.zeros((Z,3))
c3a_r=np.zeros((Z,3));c3u_r=np.zeros((Z,3));al3_r=np.zeros((Z,3));c3_r=np.zeros((Z,3))
u1_r=np.zeros((Z,3));u2_r=np.zeros((Z,3));u3_r=np.zeros((Z,3))
sec_labels={0:'вт',1:'ср',2:'к'}

for i in range(Z):
    v=(i==0)
    Dk1=Dk_arr[i];Dvt1=Dvt_arr[i];Dk3=Dk_arr[i+1];Dvt3=Dvt_arr[i+1]
    Dk2=0.5*(Dk1+Dk3);Dvt2=0.5*(Dvt1+Dvt3)
    Dsr1=np.sqrt((Dk1**2+Dvt1**2)/2.0);Dsr3=np.sqrt((Dk3**2+Dvt3**2)/2.0)
    u_vt1=omega*Dvt1/2;u_sr1=omega*Dsr1/2;u_k1=omega*Dk1/2
    u_vt2=omega*Dvt2/2;u_sr2=omega*np.sqrt((Dk2**2+Dvt2**2)/2.0)/2;u_k2=omega*Dk2/2
    u_vt3=omega*Dvt3/2;u_sr3=omega*Dsr3/2;u_k3=omega*Dk3/2
    u1_r[i]=[u_vt1,u_sr1,u_k1];u2_r[i]=[u_vt2,u_sr2,u_k2];u3_r[i]=[u_vt3,u_sr3,u_k3]
    u_ref=u_sr3
    c1a_base=ca1_dim[i];c2a_base=ca2_dim[i]
    c3a_base=ca1_dim[i+1] if i<Z-1 else ca2_dim[i]
    HT=HT_dim[i];R=R_dist[i]
    u_s1=[u_vt1,u_ref,u_k1];u_s2=[u_vt2,u_ref,u_k2];u_s3=[u_vt3,u_ref,u_k3]
    
    if v:
        print(f"=== СТУПЕНЬ {i+1}: три сечения ===")
        print(f"  U_вт = {u_vt1:.1f},  U_ср = {u_sr1:.1f},  U_к = {u_k1:.1f} м/с")
        print(f"  U_ref(=U_ср3) = {u_ref:.1f},  H_T = {HT:.0f},  R = {R}")
    
    for j in range(3):
        uu1,uu2,uu3=u_s1[j],u_s2[j],u_s3[j]
        c1a_sq=c1a_base**2-2.0*(1.0-R)**2*(uu1**2-u_ref**2)
        c1a_r[i,j]=np.sqrt(max(c1a_sq,1.0))
        c1u_r[i,j]=uu1*(1.0-R)-HT/(2.0*uu1)
        al1_r[i,j]=np.arctan2(c1a_r[i,j],c1u_r[i,j])
        c1_r[i,j]=c1a_r[i,j]/max(np.sin(al1_r[i,j]),0.01)
        w1a_r[i,j]=c1a_r[i,j];w1u_r[i,j]=uu1-c1u_r[i,j]
        bt1_r[i,j]=np.arctan2(w1a_r[i,j],w1u_r[i,j])
        w1_r[i,j]=w1a_r[i,j]/max(np.sin(bt1_r[i,j]),0.01)
        c2a_sq=c2a_base**2-2.0*(1.0-R)**2*(uu2**2-u_ref**2)
        c2a_r[i,j]=np.sqrt(max(c2a_sq,1.0))
        c2u_r[i,j]=uu2*(1.0-R)+HT/(2.0*uu2)
        al2_r[i,j]=np.arctan2(c2a_r[i,j],c2u_r[i,j])
        c2_r[i,j]=c2a_r[i,j]/max(np.sin(al2_r[i,j]),0.01)
        w2a_r[i,j]=c2a_r[i,j];w2u_r[i,j]=uu2-c2u_r[i,j]
        bt2_r[i,j]=np.arctan2(w2a_r[i,j],w2u_r[i,j])
        w2_r[i,j]=w2a_r[i,j]/max(np.sin(bt2_r[i,j]),0.01)
        c3a_sq=c3a_base**2-2.0*(1.0-R)**2*(uu3**2-u_ref**2)
        c3a_r[i,j]=np.sqrt(max(c3a_sq,1.0))
        c3u_r[i,j]=uu3*(1.0-R)-HT/(2.0*uu3)
        al3_r[i,j]=np.arctan2(c3a_r[i,j],c3u_r[i,j])
        c3_r[i,j]=c3a_r[i,j]/max(np.sin(al3_r[i,j]),0.01)
        
        if v:
            sl=sec_labels[j]
            print(f"\n  --- Сечение {sl} (U={uu1:.1f}) ---")
            print(f"  (107) ca1_{sl} = sqrt(ca1_ср^2 - 2*(1-R)^2*(U_{sl}^2-U_ref^2))")
            print(f"       = sqrt({c1a_base:.1f}^2 - 2*(1-{R})^2*({uu1:.1f}^2-{u_ref:.1f}^2))")
            print(f"       = sqrt({c1a_base**2:.1f} - {2*(1-R)**2*(uu1**2-u_ref**2):.1f}) = {c1a_r[i,j]:.1f}")
            print(f"  (110) cu1_{sl} = U*(1-R) - H_T/(2U) = {uu1:.1f}*(1-{R}) - {HT:.0f}/(2*{uu1:.1f})")
            print(f"       = {uu1*(1-R):.1f} - {HT/(2*uu1):.1f} = {c1u_r[i,j]:.1f}")
            print(f"  alpha1 = {np.degrees(al1_r[i,j]):.1f},  c1 = {c1_r[i,j]:.1f}")
            print(f"  beta1 = {np.degrees(bt1_r[i,j]):.1f},  w1 = {w1_r[i,j]:.1f}")
            print(f"  (108) ca2_{sl} = {c2a_r[i,j]:.1f},  (111) cu2_{sl} = {c2u_r[i,j]:.1f}")
            print(f"  alpha2 = {np.degrees(al2_r[i,j]):.1f},  c2 = {c2_r[i,j]:.1f},  w2 = {w2_r[i,j]:.1f}")
            print(f"  (109) ca3_{sl} = {c3a_r[i,j]:.1f},  (112) cu3_{sl} = {c3u_r[i,j]:.1f}")

### Числа Маха

In [ ]:
Mw1_per=np.zeros(Z);Mc2_per=np.zeros(Z);Mw1_hub=np.zeros(Z);Mc2_hub=np.zeros(Z)
for i in range(Z):
    k_i=k_st[i];k_i3=k_air(T1s[i+1],Rg)
    for j,(Mw_a,Mc_a) in enumerate([(Mw1_hub,Mc2_hub),(Mw1_sr,Mc2_sr),(Mw1_per,Mc2_per)]):
        l1j=c1_r[i,j]/acr1[i];T1sj=T1s[i]*tau_lam(l1j,k_i)
        if T1sj>0: Mw_a[i]=w1_r[i,j]/np.sqrt(k_i*Rg*T1sj)
        l2j=c2_r[i,j]/acr3[i];T2sj=T1s[i+1]*tau_lam(l2j,k_i3)
        if T2sj>0: Mc_a[i]=c2_r[i,j]/np.sqrt(k_i3*Rg*T2sj)

# Подробно для 1-й ступени
i=0; k_i=k_st[0]; k_i3=k_air(T1s[1],Rg)
for j,sl in [(0,'вт'),(1,'ср'),(2,'к')]:
    l1j=c1_r[i,j]/acr1[i]; T1sj=T1s[i]*tau_lam(l1j,k_i)
    a1j=np.sqrt(k_i*Rg*T1sj)
    print(f"  Mw1_{sl}: lam_c1={l1j:.4f}, T_stat={T1sj:.1f}, a={a1j:.1f}, w1={w1_r[i,j]:.1f}")
    print(f"    = {w1_r[i,j]:.1f}/{a1j:.1f} = {w1_r[i,j]/a1j:.4f}")

print()
header=f"{'':>10}"; 
for i in range(Z): header+=f" {'Ст.'+str(i+1):>8}"
print(header)
for lbl,arr in [("Mw1_к",Mw1_per),("Mw1_ср",Mw1_sr),("Mw1_вт",Mw1_hub),
                ("Mc2_к",Mc2_per),("Mc2_ср",Mc2_sr),("Mc2_вт",Mc2_hub)]:
    s=f"{lbl:<10}"
    for i in range(Z): s+=f" {arr[i]:>8.4f}"
    print(s)

## 4. Профилирование — формулы (151)-(218)

Подробно для 1-й ступени, среднее сечение.

In [ ]:
def eps_bt1(b2d):
    return -2.1909e-5*b2d**3+3.7957e-3*b2d**2+0.262104*b2d-4.49505
def eps_ratio(bt):
    return 0.085078*bt**3-0.562903*bt**2+1.516152*bt-0.025814
def find_bt(er,b2d):
    e1=eps_bt1(b2d)
    if e1<=0: return 1.0
    try: return brentq(lambda x: eps_ratio(x)-er/e1,0.05,3.0)
    except: return 1.0

K_RK_vt=0.9;K_RK_sr=1.0;K_RK_k=1.2;K_NA_vt=0.9;K_NA_sr=1.0;K_NA_k=1.2
AR_RK=np.linspace(1.6,1.3,Z);AR_NA=np.linspace(1.6,1.3,Z)
b_RK=h_blade/AR_RK;b_NA=h_blade/AR_NA
beta1_3s=np.degrees(bt1_r);beta2_3s=np.degrees(bt2_r)
alpha2_3s=np.degrees(al2_r);alpha3_3s=np.degrees(al3_r)
eps_RK_3s=beta2_3s-beta1_3s;eps_NA_3s=alpha3_3s-alpha2_3s
bt_RK=np.array([find_bt(abs(np.degrees(eps_RK[i])),np.degrees(beta2[i])) for i in range(Z)])
bt_NA=np.array([find_bt(abs(np.degrees(eps_NA[i])),np.degrees(alpha3[i])) for i in range(Z)])
Dsr1_prof=np.sqrt((Dk_arr[:Z]**2+Dvt_arr[:Z]**2)/2.0)
Dk2_p=0.5*(Dk_arr[:Z]+Dk_arr[1:Z+1]);Dvt2_p=0.5*(Dvt_arr[:Z]+Dvt_arr[1:Z+1])
Dsr2_prof=np.sqrt((Dk2_p**2+Dvt2_p**2)/2.0)
t_RK_sr=b_RK/bt_RK;t_NA_sr=b_NA/bt_NA
Z_RK_raw=np.maximum(np.ceil(np.pi*Dsr1_prof/t_RK_sr).astype(int),10)
Z_RK_blades=np.where(Z_RK_raw%2==0,Z_RK_raw+1,Z_RK_raw)
Z_NA_raw=np.maximum(np.ceil(np.pi*Dsr2_prof/t_NA_sr).astype(int),10)
Z_NA_blades=np.where(Z_NA_raw%2==1,Z_NA_raw+1,Z_NA_raw)
Dk1_p=Dk_arr[:Z];Dvt1_p=Dvt_arr[:Z]
t_RK_k=np.pi*Dk1_p/Z_RK_blades;t_RK_sr_a=np.pi*Dsr1_prof/Z_RK_blades;t_RK_vt=np.pi*Dvt1_p/Z_RK_blades
t_NA_k=np.pi*Dk2_p/Z_NA_blades;t_NA_sr_a=np.pi*Dsr2_prof/Z_NA_blades;t_NA_vt=np.pi*Dvt2_p/Z_NA_blades
bt_RK_3s=np.column_stack([K_RK_vt*b_RK/t_RK_vt,K_RK_sr*b_RK/t_RK_sr_a,K_RK_k*b_RK/t_RK_k])
bt_NA_3s=np.column_stack([K_NA_vt*b_NA/t_NA_vt,K_NA_sr*b_NA/t_NA_sr_a,K_NA_k*b_NA/t_NA_k])
b_RK_chord_3s=np.column_stack([K_RK_vt*b_RK,K_RK_sr*b_RK,K_RK_k*b_RK])
b_NA_chord_3s=np.column_stack([K_NA_vt*b_NA,K_NA_sr*b_NA,K_NA_k*b_NA])
X_f=0.5
i_RK_3s=np.zeros((Z,3));i_NA_3s=np.zeros((Z,3))
m_RK_3s=np.zeros((Z,3));m_NA_3s=np.zeros((Z,3))
theta_RK_3s=np.zeros((Z,3));theta_NA_3s=np.zeros((Z,3))
delta_RK_3s=np.zeros((Z,3));delta_NA_3s=np.zeros((Z,3))
ups_RK_3s=np.zeros((Z,3));ups_NA_3s=np.zeros((Z,3))
Rsl_RK_3s=np.zeros((Z,3));Rsl_NA_3s=np.zeros((Z,3))

for i in range(Z):
    for j in range(3):
        i_RK_3s[i,j]=2.5*(bt_RK_3s[i,j]-1.0)
        m_RK_3s[i,j]=0.23*(2*X_f)**2+0.18-0.002*beta2_3s[i,j]
        ea=abs(eps_RK_3s[i,j])
        dn=1.0-m_RK_3s[i,j]*np.sqrt(1.0/bt_RK_3s[i,j]) if bt_RK_3s[i,j]>0.01 else 1.0
        theta_RK_3s[i,j]=(ea-i_RK_3s[i,j])/dn if abs(dn)>0.01 else ea
        delta_RK_3s[i,j]=m_RK_3s[i,j]*theta_RK_3s[i,j]*np.sqrt(1.0/bt_RK_3s[i,j]) if bt_RK_3s[i,j]>0.01 else 0
        ups_RK_3s[i,j]=0.5*theta_RK_3s[i,j]+beta1_3s[i,j]+i_RK_3s[i,j]
        sh=np.sin(np.radians(theta_RK_3s[i,j]/2.0))
        Rsl_RK_3s[i,j]=b_RK_chord_3s[i,j]/(2.0*sh) if abs(sh)>0.001 else 0
        i_NA_3s[i,j]=2.5*(bt_NA_3s[i,j]-2.0)
        m_NA_3s[i,j]=0.23*(2*X_f)**2+0.18-0.002*alpha3_3s[i,j]
        ea2=abs(eps_NA_3s[i,j])
        dn2=1.0-m_NA_3s[i,j]*np.sqrt(1.0/bt_NA_3s[i,j]) if bt_NA_3s[i,j]>0.01 else 1.0
        theta_NA_3s[i,j]=(ea2-i_NA_3s[i,j])/dn2 if abs(dn2)>0.01 else ea2
        delta_NA_3s[i,j]=m_NA_3s[i,j]*theta_NA_3s[i,j]*np.sqrt(1.0/bt_NA_3s[i,j]) if bt_NA_3s[i,j]>0.01 else 0
        ups_NA_3s[i,j]=0.5*theta_NA_3s[i,j]+alpha2_3s[i,j]+i_NA_3s[i,j]
        sh2=np.sin(np.radians(theta_NA_3s[i,j]/2.0))
        Rsl_NA_3s[i,j]=b_NA_chord_3s[i,j]/(2.0*sh2) if abs(sh2)>0.001 else 0

# Подробно: 1-я ступень, среднее сечение (j=1)
i0=0;j0=1
print("=== Ступень 1, среднее сечение, РК ===")
b2d=beta2_3s[i0,j0]; ea=abs(eps_RK_3s[i0,j0])
e1v=eps_bt1(b2d)
print(f"  (151) eps_bt1(beta2={b2d:.1f}) = {e1v:.2f}")
print(f"  eps_РК = {ea:.1f},  eps/eps_bt1 = {ea/e1v:.4f}")
print(f"  (153) b/t_РК = {bt_RK_3s[i0,j0]:.3f}")
print()
print(f"  (155) b_РК = h/AR*K = {h_blade[i0]*1000:.1f}/{AR_RK[i0]:.2f}*{K_RK_sr} = {b_RK_chord_3s[i0,j0]*1000:.1f} мм")
print(f"  (159) Z_РК = ceil(pi*D_ср/t_РК) = {Z_RK_blades[i0]}")
print(f"  (161) t_РК = pi*D_ср/Z = pi*{Dsr1_prof[i0]*1000:.1f}/{Z_RK_blades[i0]} = {t_RK_sr_a[i0]*1000:.1f} мм")
print(f"  (163) b/t_РК (уточн.) = {bt_RK_3s[i0,j0]:.3f}")
print()
print(f"  (165) i_РК = 2.5*(b/t-1) = 2.5*({bt_RK_3s[i0,j0]:.3f}-1) = {i_RK_3s[i0,j0]:.2f} град")
print(f"  (167) m_РК = 0.23*(2*{X_f})^2+0.18-0.002*{b2d:.1f} = {m_RK_3s[i0,j0]:.4f}")
print(f"  (169) theta_РК = (eps-i)/(1-m/sqrt(b/t))")
print(f"       = ({ea:.1f}-{i_RK_3s[i0,j0]:.2f})/(1-{m_RK_3s[i0,j0]:.4f}/sqrt({bt_RK_3s[i0,j0]:.3f}))")
print(f"       = {ea-i_RK_3s[i0,j0]:.2f}/{1-m_RK_3s[i0,j0]/np.sqrt(bt_RK_3s[i0,j0]):.4f} = {theta_RK_3s[i0,j0]:.1f} град")
print(f"  (171) delta_РК = m*theta/sqrt(b/t) = {m_RK_3s[i0,j0]:.4f}*{theta_RK_3s[i0,j0]:.1f}/sqrt({bt_RK_3s[i0,j0]:.3f}) = {delta_RK_3s[i0,j0]:.2f}")
print(f"  (173) chi_РК = theta/2 = {theta_RK_3s[i0,j0]/2:.1f} град")
print(f"  (175) ups_РК = chi+beta1+i = {theta_RK_3s[i0,j0]/2:.1f}+{beta1_3s[i0,j0]:.1f}+{i_RK_3s[i0,j0]:.2f} = {ups_RK_3s[i0,j0]:.1f} град")
sh0=np.sin(np.radians(theta_RK_3s[i0,j0]/2))
print(f"  (177) R_сл = b/(2*sin(chi)) = {b_RK_chord_3s[i0,j0]*1000:.1f}/(2*sin({theta_RK_3s[i0,j0]/2:.1f})) = {Rsl_RK_3s[i0,j0]*1000:.1f} мм")

print(f"\n=== Таблица профилирования (все ступени, среднее) ===")
print(f"{'Ступ':>5} {'b/t_РК':>7} {'i_РК':>7} {'th_РК':>7} {'d_РК':>7} {'ups_РК':>7} {'Rsl_РК':>8}")
for i in range(Z):
    print(f"{i+1:>5d} {bt_RK_3s[i,1]:>7.3f} {i_RK_3s[i,1]:>7.2f} {theta_RK_3s[i,1]:>7.1f} {delta_RK_3s[i,1]:>7.2f} {ups_RK_3s[i,1]:>7.1f} {Rsl_RK_3s[i,1]:>8.4f}")

## 5. Расчёт на прочность

In [ ]:
from scipy.integrate import quad
from scipy.optimize import fsolve
F_sections=np.array([401.9,372.2,357.3])*1e-6
R_sections=np.array([225.5,300.5,376.0])*1e-3
rho_blade=4450
def area_sys(p,R,F): return [p[0]+p[1]*R[0]**p[2]-F[0],p[0]+p[1]*R[1]**p[2]-F[1],p[0]+p[1]*R[2]**p[2]-F[2]]
try: a_f,b_f,c_f=fsolve(area_sys,[0.16,-0.16,5e-4],args=(R_sections,F_sections))
except: a_f,b_f,c_f=0.162,-0.162,5.445e-4
def F_blade(z): return a_f+b_f*z**c_f
R_per=R_sections[2]; omega_b=u_k/R_per
R_kor=R_sections[0]
N_kor,_=quad(lambda z1: z1*F_blade(z1),R_kor,R_per)
N_kor*=rho_blade*omega_b**2
sigma_r=N_kor/F_sections[0]
print(f"F(z) = {a_f:.4f} + {b_f:.4f}*z^{c_f:.6f}")
print(f"omega = Uk/R_per = {u_k:.2f}/{R_per:.4f} = {omega_b:.2f} рад/с")
print(f"rho = {rho_blade} кг/м3")
print(f"N(Rk) = rho*omega^2 * integral = {rho_blade}*{omega_b:.2f}^2 * {N_kor/(rho_blade*omega_b**2):.8f}")
print(f"      = {N_kor:.0f} Н")
print(f"sigma_р(Rk) = N/F = {N_kor:.0f}/{F_sections[0]*1e6:.1f}e-6 = {sigma_r/1e6:.1f} МПа")

## 6. Итоговая сводка

In [ ]:
print("="*70)
print("ИТОГОВАЯ СВОДКА")
print("="*70)
print(f"  G={G}, T*0={T0_star}, P*0={P0_star:.0f}, pi*k={pi_k_star}, n={n_rpm}")
print(f"  ca_bar_1={ca_bar_1}, d_bar_1={d_bar_1}")
print(f"  D_k={D_k*1000:.0f} мм, D_vt_1={D_vt_1*1000:.0f} мм")
print(f"  h_1={h_1*1000:.1f} мм, h_n={(Dk_arr[Z]-Dvt_arr[Z])/2*1000:.1f} мм")
print(f"  Z={Z}, Uk={u_k:.1f} м/с")
print(f"  pi*k={pi_k_calc:.2f} (цель {pi_k_star}), eta*k={eta_k_calc:.4f}")
print(f"  T*вых={T1s[Z]:.1f} К, P*вых={P1s[Z]/1000:.1f} кПа")
print(f"  Nk={N_k:.0f} кВт = {N_k/1000:.1f} МВт")
print(f"  dpi*={delta_pi:+.2f}%")